# Air Pollution Trends in Myanmar

This notebook summarizes tropospheric nitrogen dioxide (NO₂) levels in Myanmar using data from NASA's Ozone Monitoring Instrument (OMI). This analysis focuses on annual national trends across Myanmar (ADM0) and monthly/annual trends for Yangon (ADM1).

In [ ]:
from pathlib import Path
import warnings

import altair as alt
import pandas as pd

try:
    import altairtheme

    altairtheme.enable()
except ImportError:
    pass

warnings.filterwarnings("ignore")
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [23]:
PROJECT_ROOT = Path().cwd().parent.parent
DATA_PATH = PROJECT_ROOT / "data"
OMI_DIR = DATA_PATH / "AirPollution" / "omi_historic"
CONVERSION_CONSTANT = 6.02214076e19

omi_daily_adm0 = (
    pd.concat(
        [pd.read_csv(f) for f in sorted(OMI_DIR.glob("omi_no2_myanmar_*.csv"))],
        ignore_index=True,
    )
    .query('date < "2026-01-01"')
    .assign(
        date=lambda df: pd.to_datetime(df["date"]),
        NO2_mean=lambda df: df["NO2_mean"] / CONVERSION_CONSTANT,
    )
    .sort_values("date")
    .reset_index(drop=True)
)

omi_annual_adm0 = (
    omi_daily_adm0.set_index("date")
    .resample("YS")
    .agg(
        no2=("NO2_mean", "mean"),
        valid_pixels_mean=("valid_pixels", "mean"),
        n_days=("NO2_mean", "count"),
    )
)

omi_daily_yangon = (
    pd.concat(
        [
            pd.read_csv(DATA_PATH / "AirPollution" / f"omi_no2_monthly_adm1_{r}.csv")
            for r in ["2010_2020", "2021_2025"]
        ]
    )
    .assign(
        date=lambda df: pd.to_datetime(df["date"]),
        NO2_mean=lambda df: df["no2_trop_mean"] / CONVERSION_CONSTANT,
    )
    .query("ST == 'Yangon'")
)

omi_monthly_yangon = (
    omi_daily_yangon.set_index("date")
    .resample("MS")
    .agg(
        no2=("NO2_mean", "mean"),
        valid_pixels_mean=("pixel_count", "mean"),
        n_days=("NO2_mean", "count"),
    )
)

omi_annual_yangon = (
    omi_daily_yangon.set_index("date")
    .resample("YS")
    .agg(
        no2=("NO2_mean", "mean"),
        valid_pixels_mean=("pixel_count", "mean"),
        n_days=("NO2_mean", "count"),
    )
)

## Annual National NO₂ Trend

The chart below shows the mean annual tropospheric NO₂ column density across Myanmar.

In [29]:
nearest = alt.selection_point(
    on="mouseover", fields=["date"], nearest=True, empty=False
)
base = alt.Chart(omi_annual_adm0.reset_index())
line = base.mark_line(strokeWidth=2.5).encode(
    x=alt.X("date:T", title="", axis=alt.Axis(format="%Y")),
    y=alt.Y("no2:Q", title="NO₂"),
)
points = base.mark_point(filled=True, size=60).encode(
    x=alt.X("date:T"),
    y=alt.Y("no2:Q"),
)
rules = (
    base.mark_rule(color="gray")
    .encode(
        x="date:T",
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Year", format="%Y"),
            alt.Tooltip("no2:Q", title="NO₂", format=".6f"),
        ],
    )
    .add_params(nearest)
)

(line + points + rules).properties(
    width=600,
    height=300,
    title={
        "text": "Annual National NO₂ Trend in Myanmar",
        "subtitle": "Source: NASA OMI (Ozone Monitoring Instrument), 2010–2025",
    },
)

alt.LayerChart(...)

## Yangon NO₂ Trends

The charts below show the monthly and annual mean tropospheric NO₂ column density for Yangon.

In [32]:
nearest = alt.selection_point(
    on="mouseover", fields=["date"], nearest=True, empty=False
)
base = alt.Chart(omi_monthly_yangon.reset_index())
line = base.mark_line(color="steelblue").encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("no2:Q", title="NO₂"),
)
rules = (
    base.mark_rule(color="gray")
    .encode(
        x="date:T",
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Month", format="%b %Y"),
            alt.Tooltip("no2:Q", title="NO₂", format=".6f"),
        ],
    )
    .add_params(nearest)
)

(line + rules).properties(
    width=700,
    height=280,
    title={
        "text": "Monthly NO₂ in Yangon",
        "subtitle": "Source: NASA OMI, 2010–2025",
    },
)

alt.LayerChart(...)

In [33]:
nearest = alt.selection_point(
    on="mouseover", fields=["date"], nearest=True, empty=False
)
base = alt.Chart(omi_annual_yangon.reset_index())
line = base.mark_line().encode(
    x=alt.X("date:T", title="", axis=alt.Axis(format="%Y")),
    y=alt.Y("no2:Q", title="NO₂ (×10¹⁵ molecules/cm²)"),
)
points = base.mark_point(filled=True, size=60).encode(
    x=alt.X("date:T"),
    y=alt.Y("no2:Q"),
)
rules = (
    base.mark_rule(color="gray")
    .encode(
        x="date:T",
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Year", format="%Y"),
            alt.Tooltip("no2:Q", title="NO₂", format=".6f"),
        ],
    )
    .add_params(nearest)
)

(line + points + rules).properties(
    width=600,
    height=280,
    title={
        "text": "Annual NO₂ Trend in Yangon",
        "subtitle": "Source: NASA OMI, 2010–2025",
    },
)

alt.LayerChart(...)